# Fundamentals of Data Science — Case Study Notebook
**Student:** Annika Reddy  
**Roll Number:** 160124733153  
**Case Study 1:** Online Retail  
**Case Study 4:** Student Data and Performance

This notebook is prepared from the accompanying report. Upload the original datasets when prompted. The report PDF alone does not contain the raw data, so numerical results and plots cannot be reproduced until the datasets are provided.

## Setup
Run this cell first. If a library is missing in your environment, install it with `%pip install package-name`.

In [ ]:
import os, glob, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, classification_report, confusion_matrix, balanced_accuracy_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 7
print("Setup complete.")

# Case Study 1 — Online Retail

## 1. Load the transaction dataset
Expected format from the report: one basket per row, comma-separated products, no header, and up to 20 product columns. Upload `Online_retail_new.csv` to the notebook environment first.

In [ ]:
# In Google Colab, uncomment and run these lines to upload the CSV:
# from google.colab import files
# uploaded = files.upload()

retail_candidates = glob.glob("*Online*retail*.csv") + glob.glob("*online*retail*.csv")
print("Possible retail files:", retail_candidates)

# Update this path if your file has a different name.
RETAIL_PATH = "Online_retail_new.csv"

if os.path.exists(RETAIL_PATH):
    retail_raw = pd.read_csv(RETAIL_PATH, header=None, dtype=str, keep_default_na=False)
    print("Raw shape:", retail_raw.shape)
    display(retail_raw.head())
else:
    retail_raw = None
    print("Dataset not found. Upload Online_retail_new.csv, then rerun this cell.")

## 2. Clean the baskets and create a binary basket-product matrix

In [ ]:
if retail_raw is not None:
    baskets = []
    for _, row in retail_raw.iterrows():
        items = [str(x).strip().lower() for x in row.tolist() if str(x).strip()]
        # Count each product at most once in a basket
        baskets.append(sorted(set(items)))

    mlb = MultiLabelBinarizer()
    basket_matrix = mlb.fit_transform(baskets)
    basket_df = pd.DataFrame(basket_matrix, columns=mlb.classes_)

    basket_sizes = np.array([len(b) for b in baskets])
    print("Baskets:", len(baskets))
    print("Distinct products:", len(mlb.classes_))
    print("Basket-product entries:", int(basket_matrix.sum()))
    print("Mean basket size:", round(basket_sizes.mean(), 2))
    print("Standard deviation:", round(basket_sizes.std(), 2))
    print("Median basket size:", float(np.median(basket_sizes)))
    print("Minimum / maximum:", int(basket_sizes.min()), "/", int(basket_sizes.max()))
else:
    print("Run the previous cell after uploading the dataset.")

## 3. Product frequency and basket-size distribution

In [ ]:
if retail_raw is not None:
    product_counts = basket_df.sum().sort_values(ascending=False)
    product_share = 100 * product_counts / len(basket_df)
    display(pd.DataFrame({"Baskets": product_counts, "% of baskets": product_share.round(2)}).head(15))

    product_share.head(15).sort_values().plot(kind="barh", figsize=(8, 6))
    plt.title("Top 15 Products by Basket Penetration")
    plt.xlabel("Share of baskets (%)")
    plt.tight_layout()
    plt.show()

    pd.Series(basket_sizes).value_counts().sort_index().plot(kind="bar", figsize=(8, 4))
    plt.title("Basket-size Frequencies")
    plt.xlabel("Items per basket")
    plt.ylabel("Number of baskets")
    plt.tight_layout()
    plt.show()

## 4. Pairwise association: support, confidence and lift

In [ ]:
if retail_raw is not None:
    # Products appearing in at least 1% of baskets, as described in the report
    min_support_count = int(np.ceil(0.01 * len(basket_df)))
    eligible = product_counts[product_counts >= min_support_count].index.tolist()
    assoc = basket_df[eligible]
    n = len(assoc)
    rows = []
    for i, a in enumerate(eligible):
        a_count = int(assoc[a].sum())
        for b in eligible:
            if a == b:
                continue
            joint = int(((assoc[a] == 1) & (assoc[b] == 1)).sum())
            if joint >= 75:
                b_count = int(assoc[b].sum())
                support = joint / n
                confidence = joint / a_count if a_count else 0
                lift = confidence / (b_count / n) if b_count else 0
                rows.append((a, b, joint, support, confidence, lift))
    rules = pd.DataFrame(rows, columns=["If basket has", "Then also", "Joint baskets", "Support", "Confidence", "Lift"])
    display(rules.sort_values("Lift", ascending=False).head(15).round(3))

## 5. PCA and clustering comparison

In [ ]:
if retail_raw is not None:
    rng = np.random.RandomState(RANDOM_STATE)
    sample_n = min(2000, len(basket_df))
    sample_idx = rng.choice(len(basket_df), size=sample_n, replace=False)
    X = basket_df.iloc[sample_idx].to_numpy()

    pca = PCA(random_state=RANDOM_STATE)
    X_pca_all = pca.fit_transform(X)
    print("PC1 variance (%):", round(pca.explained_variance_ratio_[0] * 100, 2))
    print("PC2 variance (%):", round(pca.explained_variance_ratio_[1] * 100, 2))
    print("First 10 components cumulative variance (%):", round(pca.explained_variance_ratio_[:10].sum() * 100, 2))

    plt.figure(figsize=(7, 4))
    plt.bar(range(1, min(10, X.shape[1]) + 1), pca.explained_variance_ratio_[:10] * 100)
    plt.title("PCA Scree Plot (first 10 components)")
    plt.xlabel("Principal component")
    plt.ylabel("Variance explained (%)")
    plt.tight_layout()
    plt.show()

    # K-means sweep (Euclidean distance, consistent with sklearn KMeans)
    sweep = []
    for k in range(2, 9):
        km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
        labels = km.fit_predict(X)
        sweep.append((k, km.inertia_, silhouette_score(X, labels), davies_bouldin_score(X, labels)))
    sweep_df = pd.DataFrame(sweep, columns=["k", "Inertia", "Silhouette", "Davies-Bouldin"])
    display(sweep_df.round(3))

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(sweep_df["k"], sweep_df["Inertia"], marker="o")
    ax[0].set_title("K-means Elbow")
    ax[0].set_xlabel("k")
    ax[0].set_ylabel("Inertia")
    ax[1].plot(sweep_df["k"], sweep_df["Silhouette"], marker="o")
    ax[1].set_title("Silhouette by k")
    ax[1].set_xlabel("k")
    plt.tight_layout()
    plt.show()

    k = 5
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    km_labels = km.fit_predict(X)
    print("K-means (k=5): silhouette =", round(silhouette_score(X, km_labels), 3),
          "; Davies-Bouldin =", round(davies_bouldin_score(X, km_labels), 3))

    # Hierarchical and spectral clustering
    hier_labels = AgglomerativeClustering(n_clusters=5, linkage="complete").fit_predict(X)
    spec_labels = SpectralClustering(n_clusters=5, affinity="nearest_neighbors",
                                     n_neighbors=min(15, len(X)-1), random_state=RANDOM_STATE).fit_predict(X)
    for name, labels in [("Hierarchical", hier_labels), ("Spectral", spec_labels)]:
        print(name, "silhouette =", round(silhouette_score(X, labels), 3),
              "; Davies-Bouldin =", round(davies_bouldin_score(X, labels), 3))

    # DBSCAN using Jaccard distance on binary data
    from sklearn.metrics import pairwise_distances
    D = pairwise_distances(X, metric="jaccard")
    db_labels = DBSCAN(eps=0.5, min_samples=5, metric="precomputed").fit_predict(D)
    non_noise = db_labels != -1
    print("DBSCAN clusters (excluding noise):", len(set(db_labels[non_noise])))
    print("DBSCAN noise baskets:", int((~non_noise).sum()))
    if len(set(db_labels[non_noise])) >= 2:
        print("DBSCAN silhouette on non-noise points:", round(silhouette_score(X[non_noise], db_labels[non_noise]), 3))
else:
    print("Upload the retail dataset to run PCA and clustering.")

## Case Study 1 — Interpretation notes
Use the actual output above when writing observations. The report describes weak cluster separation; do not claim that clustering is strong unless your rerun supports that conclusion. Association lift indicates co-occurrence, not causation.

# Case Study 4 — Student Data and Performance

## 1. Load student dataset
This report appears to use the student dropout and academic success dataset, normally a semicolon-separated CSV with a `Target` column. Upload the original dataset before running the cells below.

In [ ]:
# In Google Colab, uncomment and run:
# from google.colab import files
# uploaded = files.upload()

student_candidates = glob.glob("*student*.csv") + glob.glob("*Student*.csv") + glob.glob("*dropout*.csv")
print("Possible student files:", student_candidates)

STUDENT_PATH = "student_data.csv"  # Change this to the exact uploaded filename.

if os.path.exists(STUDENT_PATH):
    students = pd.read_csv(STUDENT_PATH, sep=";", encoding="utf-8-sig")
    students.columns = students.columns.str.strip()
    print("Dataset shape:", students.shape)
    print("Missing cells:", int(students.isna().sum().sum()))
    print("Duplicate rows:", int(students.duplicated().sum()))
    display(students.head())
else:
    students = None
    print("Dataset not found. Upload the student dataset and update STUDENT_PATH.")

## 2. Outcome distribution and exploratory analysis

In [ ]:
if students is not None:
    target_col = "Target" if "Target" in students.columns else students.columns[-1]
    display(students[target_col].value_counts().rename_axis("Outcome").to_frame("Students"))
    (students[target_col].value_counts(normalize=True) * 100).plot(kind="bar", figsize=(7, 4))
    plt.title("Distribution of Final Outcome")
    plt.ylabel("Percentage (%)")
    plt.tight_layout()
    plt.show()

    for col in ["Age at enrollment", "Admission grade", "Curricular units 1st sem (grade)"]:
        if col in students.columns:
            print("\n", col)
            display(students[col].describe().to_frame().T)

## 3. Compare first-semester grades across outcomes

In [ ]:
if students is not None:
    grade_col = "Curricular units 1st sem (grade)"
    target_col = "Target"
    if grade_col in students.columns and target_col in students.columns:
        display(students.groupby(target_col)[grade_col].agg(["count", "mean", "median", "std"]).round(3))
        students.boxplot(column=grade_col, by=target_col, figsize=(8, 5))
        plt.title("First-semester Grade by Final Outcome")
        plt.suptitle("")
        plt.xlabel("Final outcome")
        plt.ylabel("First-semester grade")
        plt.tight_layout()
        plt.show()

        from scipy.stats import kruskal
        groups = [g[grade_col].dropna().values for _, g in students.groupby(target_col)]
        if len(groups) >= 2:
            H, p = kruskal(*groups)
            n = sum(len(g) for g in groups)
            k = len(groups)
            epsilon_sq = max(0, (H - k + 1) / (n - k))
            print(f"Kruskal-Wallis H={H:.3f}, p-value={p:.4g}, epsilon-squared={epsilon_sq:.3f}")

## 4. Train and evaluate a multiclass logistic regression model

In [ ]:
if students is not None:
    target_col = "Target"
    y = students[target_col].astype(str)
    X = students.drop(columns=[target_col]).copy()

    # Remove second-semester columns to align with the report's early-prediction setup.
    second_sem_cols = [c for c in X.columns if "2nd sem" in c.lower() or "2nd semester" in c.lower()]
    X = X.drop(columns=second_sem_cols, errors="ignore")

    # Use numeric columns directly and one-hot encode categorical/text columns.
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    model = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(
            multi_class="multinomial", max_iter=3000, C=0.5,
            class_weight="balanced", random_state=RANDOM_STATE
        ))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("Accuracy:", round(accuracy_score(y_test, pred), 3))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_test, pred), 3))
    print("Macro F1:", round(f1_score(y_test, pred, average="macro"), 3))
    print("\nClass-wise report:")
    print(classification_report(y_test, pred, zero_division=0))

    labels = model.named_steps["classifier"].classes_
    cm = confusion_matrix(y_test, pred, labels=labels)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title("Confusion Matrix — Class-weighted Logistic Regression")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(range(len(labels)), labels, rotation=25)
    plt.yticks(range(len(labels)), labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.tight_layout()
    plt.show()
else:
    print("Upload the student dataset to train the model.")

## Final checklist
- [ ] Upload the original online-retail CSV and update `RETAIL_PATH`.
- [ ] Upload the original student CSV and update `STUDENT_PATH`.
- [ ] Run all cells from top to bottom.
- [ ] Review the outputs and update the written interpretations to match the actual results.
- [ ] Save/download this notebook as an `.ipynb` file.
- [ ] Upload the `.ipynb` file to the GitHub repository and submit the repository URL with the report PDF.

**Important:** This notebook is a reproducible starting point based on the report's described methods. Exact reproduction of every reported number requires the original datasets and, potentially, the exact original preprocessing and parameter choices.